# Preparacion de datos

In [278]:
import pandas as pd

### Importando archivo CSV
---

In [279]:
# Importo los datos del archvio CSV

df = pd.read_csv(
    "./datos/01-raw/Tarjetas_de_crédito_y_débito_20260802.csv",
    sep=';',
    quotechar='"',
    encoding='utf-8-sig')

In [280]:
df.sample(2)

,TIPOENTIDAD,CODIGOENTIDAD,NOMBREENTIDAD,FECHACORTE,COD_UCA,NOMBRE_UCA,SUBCUENTA,DESCRIPCION,PERSONA_NATURAL,PERSONA_JURIDICA,TOTAL_TARJETAS
13603,118,3,SOCIEDAD AL SERVICIO DE LA TECNOLOGÍA Y SISTEMATIZACIÓN BANCARIA TECNIBANCA S.A.,31/10/2024,7,ADMINISTRADORAS DE SISTEMAS DE PAGO DE BAJO VALOR,105,Número de participantes,0,0,31
8443,1,7,BANCOLOMBIA S.A.,31/07/2024,6,TARJETA DEBITO,10,Número total de tarjetas débito vigentes durante el mes,0,0,375.188


In [281]:
# Creo una copia del dataframe original
df_clean = df.copy()

### Cambio de tipo de dato en el campo de fecha "FECHACORTE"
---

In [282]:
df_clean["FECHACORTE"] = pd.to_datetime(df_clean["FECHACORTE"], dayfirst=True)

In [283]:
df_clean["FECHACORTE"].min()

Timestamp('2024-01-31 00:00:00')

In [284]:
df_clean["FECHACORTE"].max()

Timestamp('2026-05-31 00:00:00')

### Cambio de tipo de datos en valores numericos ["PERSONA_NATURAL", "PERSONA_JURIDICA", "TOTAL_TARJETAS"]
---

In [285]:
dinero_cols = ["PERSONA_NATURAL", "PERSONA_JURIDICA", "TOTAL_TARJETAS"]

df_clean[dinero_cols] = (
    df_clean[dinero_cols]
    .replace("\.", "", regex=True)
    .replace(",", ".", regex=True)
)

In [286]:

df_clean[dinero_cols] = df_clean[dinero_cols].apply(pd.to_numeric, errors = "coerce")

### Eliminar espacios y normalizacion texto ["NOMBREENTIDAD", "NOMBRE_UCA", "DESCRIPCION"]
---

In [287]:
pd.set_option("display.max_colwidth", None)

# Agrupo por codigo de entidad para ver que categorias pertenencen a un mismo codigo
df_clean.groupby("CODIGOENTIDAD")["NOMBREENTIDAD"].apply(lambda x: sorted(set(x))).reset_index()

,CODIGOENTIDAD,NOMBREENTIDAD
0,1,"[BANCO DE BOGOTA S. A., COOPERATIVA FINANCIERA DE ANTIOQUIA CFA, CREDIBANCO S.A. PUDIENDO SIN PERDER SU NATURALEZA UTILIZAR LA SIGLA CREDIBANCO]"
1,2,"[BANCO POPULAR S.A., JFK COOPERATIVA FINANCIERA]"
2,3,"[COOFINEP COOPERATIVA FINANCIERA, SOCIEDAD AL SERVICIO DE LA TECNOLOGÍA Y SISTEMATIZACIÓN BANCARIA TECNIBANCA S.A.]"
3,4,[COTRAFA FINANCIERA]
4,5,"[CONFIAR COOPERATIVA FINANCIERA, REDEBAN S.A.]"
5,6,"[ITAU CORPBANCA COLOMBIA S.A., VISIONAMOS SISTEMA DE PAGO COOPERATIVO]"
6,7,[BANCOLOMBIA S.A.]
7,8,[VISA COLOMBIA SUPPORT SERVICES SOCIEDAD ANONIMA]
8,9,"[CITIBANK - COLOMBIA S.A., MASTERCARD COLOMBIA ADMINISTRADORA S.A.]"
9,12,[BANCO GNB SUDAMERIS S.A.]


In [288]:
cat_text_cols = ["NOMBREENTIDAD", "NOMBRE_UCA", "DESCRIPCION"]

# normalizo el texto en variables categoricas
df_clean[cat_text_cols] = df_clean[cat_text_cols].apply(
    lambda col: (
        col.str.replace('"', '', regex=False)   # quitar comillas dobles
           .str.replace("'", "", regex=False)   # quitar comillas simples
           .str.replace(".", "", regex=False)   # quitar puntos
           .str.replace(",", "", regex=False)   # quitar comas ← NUEVO
           .str.replace("-", "", regex=False)   # quitar guiones
           .str.replace("   ", " ", regex=False)# triple espacio -> uno
           .str.replace("  ", " ", regex=False) # doble espacio -> uno
           .str.replace(" ", "-", regex=False)  # espacio -> guion
           .str.upper()                         # mayúsculas
           .str.strip()                         # limpiar espacios al inicio/fin
    )
)

In [289]:
df_clean["NOMBREENTIDAD"].dtype

dtype('O')

#### Revision de categorias en entidad


In [290]:
pd.set_option("display.max_colwidth", None)


df_clean.groupby("CODIGOENTIDAD")["NOMBREENTIDAD"].apply(lambda x: sorted(set(x))).reset_index()

,CODIGOENTIDAD,NOMBREENTIDAD
0,1,"[BANCO-DE-BOGOTA-S-A, COOPERATIVA-FINANCIERA-DE-ANTIOQUIA-CFA, CREDIBANCO-SA-PUDIENDO-SIN-PERDER-SU-NATURALEZA-UTILIZAR-LA-SIGLA-CREDIBANCO]"
1,2,"[BANCO-POPULAR-SA, JFK-COOPERATIVA-FINANCIERA]"
2,3,"[COOFINEP-COOPERATIVA-FINANCIERA, SOCIEDAD-AL-SERVICIO-DE-LA-TECNOLOGÍA-Y-SISTEMATIZACIÓN-BANCARIA-TECNIBANCA-SA]"
3,4,[COTRAFA-FINANCIERA]
4,5,"[CONFIAR-COOPERATIVA-FINANCIERA, REDEBAN-SA]"
5,6,"[ITAU-CORPBANCA-COLOMBIA-SA, VISIONAMOS-SISTEMA-DE-PAGO-COOPERATIVO]"
6,7,[BANCOLOMBIA-SA]
7,8,[VISA-COLOMBIA-SUPPORT-SERVICES-SOCIEDAD-ANONIMA]
8,9,"[CITIBANK-COLOMBIA-SA, MASTERCARD-COLOMBIA-ADMINISTRADORA-SA]"
9,12,[BANCO-GNB-SUDAMERIS-SA]


In [291]:
"""
Se normaliza nombres de entidades que presentan errores o no son claros
"""

df_clean.loc[df_clean["NOMBREENTIDAD"]=='BOLD-CF-BOLD-CF-SA-O-BOLD-CF', "NOMBREENTIDAD"] = "BOLD-CF"

df_clean.loc[df_clean["NOMBREENTIDAD"]=='NU-O-NU-FINANCIERA', "NOMBREENTIDAD"] = "NU"

In [292]:
# Agrupo por codigo de entidad para ver que categorias pertenencen a un mismo codigo
df_clean.groupby("CODIGOENTIDAD")["NOMBREENTIDAD"].apply(lambda x: sorted(set(x))).reset_index()

,CODIGOENTIDAD,NOMBREENTIDAD
0,1,"[BANCO-DE-BOGOTA-S-A, COOPERATIVA-FINANCIERA-DE-ANTIOQUIA-CFA, CREDIBANCO-SA-PUDIENDO-SIN-PERDER-SU-NATURALEZA-UTILIZAR-LA-SIGLA-CREDIBANCO]"
1,2,"[BANCO-POPULAR-SA, JFK-COOPERATIVA-FINANCIERA]"
2,3,"[COOFINEP-COOPERATIVA-FINANCIERA, SOCIEDAD-AL-SERVICIO-DE-LA-TECNOLOGÍA-Y-SISTEMATIZACIÓN-BANCARIA-TECNIBANCA-SA]"
3,4,[COTRAFA-FINANCIERA]
4,5,"[CONFIAR-COOPERATIVA-FINANCIERA, REDEBAN-SA]"
5,6,"[ITAU-CORPBANCA-COLOMBIA-SA, VISIONAMOS-SISTEMA-DE-PAGO-COOPERATIVO]"
6,7,[BANCOLOMBIA-SA]
7,8,[VISA-COLOMBIA-SUPPORT-SERVICES-SOCIEDAD-ANONIMA]
8,9,"[CITIBANK-COLOMBIA-SA, MASTERCARD-COLOMBIA-ADMINISTRADORA-SA]"
9,12,[BANCO-GNB-SUDAMERIS-SA]


### Revision de categorias en "NOMBRE_UCA"
---

In [293]:
df_clean["COD_UCA"].unique()

array([1, 2, 5, 6, 4, 3, 7])

In [294]:
df_clean["NOMBRE_UCA"].unique()

array(['CREDIBANCOVISA', 'MASTERCARD', 'OTRAS-TARJETAS-DE-CREDITO',
       'TARJETA-DEBITO', 'AMERICAN-EXPRESS', 'DINERS',
       'ADMINISTRADORAS-DE-SISTEMAS-DE-PAGO-DE-BAJO-VALOR'], dtype=object)

### Revision de categorias en "DESCRIPCION"
---

In [295]:
df_clean["DESCRIPCION"].unique()

array(['NÚMERO-TOTAL-DE-TARJETAS-DE-CRÉDITO-VIGENTES-A-LA-FECHA-DE-CORTE',
       'NÚMERO-TOTAL-DE-TARJETAS-DE-CRÉDITO-VIGENTES-DURANTE-EL-MES',
       'NÚMERO-TOTAL-DE-TARJETAS-DE-CRÉDITO-CANCELADAS',
       'NÚMERO-TOTAL-DE-TARJETAS-DE-CRÉDITOS-BLOQUEADAS-TEMPORALMENTE',
       'NÚMERO-DE-TRANSACCIONES-POR-COMPRAS-A-NIVEL-NACIONAL-CON-TARJETA-DE-CRÉDITO',
       'NÚMERO-DE-TRANSACCIONES-POR-AVANCES-A-NIVEL-NACIONAL-CON-TARJETA-DE-CRÉDITO',
       'NÚMERO-DE-TRANSACCIONES-POR-COMPRAS-EN-EL-EXTERIOR-CON-TARJETA-DE-CRÉDITO',
       'NÚMERO-DE-TRANSACCIONES-POR-AVANCES-EN-EL-EXTERIOR-CON-TARJETA-DE-CRÉDITO',
       'MONTO-DE-LAS-TRANSACCIONES-POR-COMPRAS-CON-TARJETA-DE-CRÉDITO-A-NIVEL-NACIONAL',
       'MONTO-DE-LAS-TRANSACCIONES-POR-AVANCES-CON-TARJETA-DE-CRÉDITO-A-NIVEL-NACIONAL',
       'MONTO-DE-LAS-TRANSACCIONES-POR-COMPRAS-EN-EL-EXTERIOR-CON-TARJETA-DE-CRÉDITO',
       'MONTO-DE-LAS-TRANSACCIONES-POR-AVANCES-EN-EL-EXTERIOR-CON-TARJETA-DE-CRÉDITO',
       'MONTO-DE-LOS-INTERESES-COR

### Calidad de datos
---

In [296]:
# Conteo de nulos por columna
df_clean.isnull().sum()

TIPOENTIDAD         0
CODIGOENTIDAD       0
NOMBREENTIDAD       0
FECHACORTE          0
COD_UCA             0
NOMBRE_UCA          0
SUBCUENTA           0
DESCRIPCION         0
PERSONA_NATURAL     0
PERSONA_JURIDICA    0
TOTAL_TARJETAS      0
dtype: int64

In [297]:
# Ver si hay algún nulo en todo el DataFrame
df_clean.isnull().any().any()

np.False_

In [298]:
# Ver si hay filas duplicadas
df_clean.duplicated().sum()

np.int64(0)

In [299]:
# Mostrar las filas duplicadas
df_clean[df_clean.duplicated()]

,TIPOENTIDAD,CODIGOENTIDAD,NOMBREENTIDAD,FECHACORTE,COD_UCA,NOMBRE_UCA,SUBCUENTA,DESCRIPCION,PERSONA_NATURAL,PERSONA_JURIDICA,TOTAL_TARJETAS


### Creacion de columnas adicionales
---

In [300]:
import locale
locale.setlocale(locale.LC_TIME, "es_CO.UTF-8")

'es_CO.UTF-8'

In [301]:
# columna adicional de año
df_clean["AÑO"] = df_clean["FECHACORTE"].dt.year

In [302]:
# Mes (número)
df_clean["MES"] = df_clean["FECHACORTE"].dt.month

In [303]:
# Nombre del mes
df_clean["MES_NOMBRE"] = df_clean["FECHACORTE"].dt.strftime("%B")

In [304]:
# Día del mes
df_clean["DIA"] = df_clean["FECHACORTE"].dt.day

### Cambio de tipo de dato en campos faltantes
---

In [305]:
cat_cols = ["TIPOENTIDAD", "CODIGOENTIDAD", "NOMBREENTIDAD", "COD_UCA", "NOMBRE_UCA", "SUBCUENTA", "DESCRIPCION", "AÑO", "MES", "MES_NOMBRE", "DIA"]

df_clean[cat_cols] = df_clean[cat_cols].astype("category")

### Revision general del resultado del pocesameinto
---

In [306]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39888 entries, 0 to 39887
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   TIPOENTIDAD       39888 non-null  category      
 1   CODIGOENTIDAD     39888 non-null  category      
 2   NOMBREENTIDAD     39888 non-null  category      
 3   FECHACORTE        39888 non-null  datetime64[ns]
 4   COD_UCA           39888 non-null  category      
 5   NOMBRE_UCA        39888 non-null  category      
 6   SUBCUENTA         39888 non-null  category      
 7   DESCRIPCION       39888 non-null  category      
 8   PERSONA_NATURAL   39888 non-null  float64       
 9   PERSONA_JURIDICA  39888 non-null  float64       
 10  TOTAL_TARJETAS    39888 non-null  float64       
 11  AÑO               39888 non-null  category      
 12  MES               39888 non-null  category      
 13  MES_NOMBRE        39888 non-null  category      
 14  DIA               3988

In [307]:
df_clean.sample(2)

,TIPOENTIDAD,CODIGOENTIDAD,NOMBREENTIDAD,FECHACORTE,COD_UCA,NOMBRE_UCA,SUBCUENTA,DESCRIPCION,PERSONA_NATURAL,PERSONA_JURIDICA,TOTAL_TARJETAS,AÑO,MES,MES_NOMBRE,DIA
14082,1,23,BANCO-DE-OCCIDENTE-SA,2024-11-30,2,MASTERCARD,35,NÚMERO-DE-TRANSACCIONES-POR-COMPRAS-EN-EL-EXTERIOR-CON-TARJETA-DE-CRÉDITO,63062.0,9481.0,7.254300e+04,2024,11,noviembre,30
10079,1,30,BANCO-CAJA-SOCIAL-SA,2024-08-31,6,TARJETA-DEBITO,45,INGRESOS-POR-TARIFA-INTERBANCARIA-DE-INTERCAMBIO-TII-POR-TARJETA-DÉBITO-VISA,0.0,0.0,3.257415e+09,2024,8,agosto,31


In [308]:
df_clean.to_parquet("./datos/02-processed/tarjetas_limpias.parquet", index=False)

In [309]:
df_clean.to_csv("./datos/02-processed/tarjetas_limpias.csv", index=False)